# 03 pycavi_run

This is a wrapper script to the mefpp engine to reduce the data that has been partioned and associated with the scripts

*pymefpp_partition_calibration
*pymefpp_partition_science
*pymefpp_associate

After running those scripts we end up with a  directory named reduced that contains the following tree structure:

* reduced/calibs
        * /bpm
        * /bias
        * /dome
        * /sky
    * /science
    
Below science we have all the science partition to be reduced.

This script will:

* take a science partition at a time
* execute the reduction cascade

The idea is to identify all biases, dome flats, sky flats that are needed for the full reduction of that science partitions, and only these, and reduce them. This permits a piecemeal approach to the work.


        
 

In [1]:
from os import listdir
from os.path import isfile, join
import glob
from astropy.io import fits
import math
import operator
import copy
import os
import shutil
from datetime import datetime
from pprint import pprint

# here we import the local mefpp_run library
import sys
sys.path.insert(0, os.getcwd())
sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_associate_reduction as mar
from pycavi import pycavi_tools
from pycavi import pycavi_instrument


This will be executed when the package is imported.


In [2]:
# where to find mefpp module
mefpp_dir = "/opt/local/eclipse/"

# Reading configuration file
#
# Asume that all notebooks are in a directory where the data reduction will
# be carried out from. It will contain initially all python libraries and 
# a the config subdirectory.
#
# debug_mode: True
# log_level: info
# observatory: Paranal
# telescope: VST
# instrument: OmegaCam
# file_prefix: OMEGA
# raw_data_dir: /Users/fselman/Pictures/OCAM/CPPup/raw/
# project_dir: /Users/fselman/Pictures/OCAM/CPPup2/
# reduced_calib_dir: /Users/fselman/Pictures/OCAM/CPPup2/calib/
# reduced_science_dir: /Users/fselman/Pictures/OCAM/CPPup2/science/
# project_logs_dir: /Users/fselman/Pictures/OCAM/CPPup2/logs/
# sex_dir_name: Sextractor_dir
# scamp_dir_name: Scamp_dir
# swarp_dir_name: Swarp_dir
# cal_dir_name: Calibration_dir
#
config_dir = os.getcwd() + '/' + '../config/'
config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
raw_data_dir = config['Directories']['raw_data_dir']
project_dir = config['Directories']['project_dir']
static_cal_dir = config['Directories']['static_calib_dir']
reduced_cal_dir = config['Directories']['reduced_calib_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
reduced_science_dir = config['Directories']['project_dir']     # Note that for this notebook we set the reduced science dir to the root project dir.
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

os.chdir(project_dir)
global cwd
cwd = os.getcwd()
print('cwd: ', cwd)

# depending on size of data set this line might be very slow
science_list = glob.glob(reduced_science_dir+'**/science.list', recursive=True)
print('number of science partitions: ', len(science_list))

std_list = glob.glob(reduced_std_dir+'**/std.list', recursive=True)
print('number of standard star partitions: ', len(std_list))


config_dir: /Users/fselman/Software/Pycavi/notebooks/../config/ instrument: omegacam
MEF.BIAS.NAME.O: OCAM_MB
MEF.BPM.NAME.O: OCAM_BPM
MEF.DOMEFF.NAME.O: OCAM_DOMEFF
MEF.SKYFF.NAME.O: OCAM_SKYFF
MEF.MFF.NAME.O: OCAM_MFF
cwd:  /Users/fselman/Pictures/OCAM/MultiObject
number of science partitions:  4
number of standard star partitions:  3


In [3]:
# associations for the science frame 
sci_partitions = []
for science_file in science_list:
    sci_partitions.append(mar.ExposuresPartition(science_file, 'SCIENCE', instrument))

std_partitions = []
for std_file in std_list:
    std_partitions.append(mar.ExposuresPartition(std_file, 'STD', instrument))

In [4]:
# select data to reduce
print('--------------------- science frame associations -------------------')
science_partition_list = sci_partitions+std_partitions
i = 0
for partition in science_partition_list:
    print(i, partition.base_path)
    i+=1

--------------------- science frame associations -------------------
0 /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
1 /Users/fselman/Pictures/OCAM/MultiObject/science/M67/NB_659/2014-12-02T07:04:51
2 /Users/fselman/Pictures/OCAM/MultiObject/science/M67/r_SDSS/2014-12-02T06:51:24
3 /Users/fselman/Pictures/OCAM/MultiObject/science/M67/g_SDSS/2014-12-02T06:32:53
4 /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/r_SDSS/2014-12-02T05:57:25
5 /Users/fselman/Pictures/OCAM/MultiObject/std/SA101/g_SDSS/2014-12-02T05:57:25
6 /Users/fselman/Pictures/OCAM/MultiObject/std/SA98/NB_659/2014-12-02T07:31:14


In [5]:

# Selection and association was done in notebook 01.
# Here we rebuild the association in memory and reduce.
#
# Note that the loop is organize to reduce a science partition
# and its requirements at a time. For some projects the user
# might prefer to reduce all calibrations first, and then the science.
# For example, to check the quality of the calibrations before
# proceeding with the science.
#
# For testing you might want to select a single partition here
# example: 
#
#          science_partition_list = [sci_partitions[3]]
#
#          science_partition_list = sci_partitions+std_partitions

science_partition_list = sci_partitions+std_partitions
for scipart in science_partition_list:

    bias_part_list = []
    dome_part_list = []
    sky_part_list = []
    
    sci_part_list = []

    # -------------- DATA ASSOCIATION SECTION ---------------
    print('\n\n#---------------------- DATA ASSOCIATION TREE -------------------------')
    print('-------------------------------------------------------------------------')
    print('scipart: '+scipart.base_path)
    sci_part_list.append(scipart)
    
    mar.associateBPM(scipart, scipart, instrument)
    print('        bpm part:  '+scipart.bpmPartition.base_path)
    mar.associateBias(scipart, scipart, instrument)
    print('        bias part:  '+scipart.biasPartition.base_path)
    bias_part_list.append(scipart.biasPartition)
    mar.associateMFF(scipart, scipart, instrument)
    print('        MFF  part:  '+scipart.mffPartition.base_path)

    # associations for the MFF frame
    mffpart = scipart.mffPartition
    print('         mffpart: '+mffpart.base_path)
    mar.associateBPM(mffpart, scipart, instrument)
    print('                  bpm  part:  '+mffpart.bpmPartition.base_path)
    mar.associateBias(mffpart, scipart, instrument)
    print('                  bias part:  '+mffpart.biasPartition.base_path)
    bias_part_list.append(mffpart.biasPartition)
    mar.associateDome(mffpart, scipart, instrument)
    print('                  dome part:  '+mffpart.domePartition.base_path)
    dome_part_list.append(mffpart.domePartition)
    mar.associateSky(mffpart, scipart, instrument)
    print('                   sky part:  '+mffpart.skyPartition.base_path)
    sky_part_list.append(mffpart.skyPartition)

    # associations for the SKY frame
    skypart = mffpart.skyPartition
    print('  ---> skypart: '+skypart.base_path)
    mar.associateBPM(skypart, scipart, instrument)
    print('                  bpm  part:  '+skypart.bpmPartition.base_path)
    mar.associateBias(skypart, scipart, instrument)
    print('                  bias part:  '+skypart.biasPartition.base_path)
    bias_part_list.append(skypart.biasPartition)
    mar.associateDome(skypart, scipart, instrument)
    print('                  dome part:  '+skypart.domePartition.base_path)
    dome_part_list.append(skypart.domePartition)

    # associations for the dome frame
    domepart = mar.ExposuresPartition(skypart.domePartition.frame, 'FLAT,DOME', instrument)
    print('  ---> domepart: '+domepart.base_path)
    mar.associateBPM(domepart, scipart, instrument)
    print('                  bpm  part:  '+domepart.bpmPartition.base_path)
    mar.associateBias(domepart, scipart, instrument)
    print('                  bias part:  '+domepart.biasPartition.base_path)
    bias_part_list.append(domepart.biasPartition)
    print ('%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%')
    
    # --------------------- DATA REDUCTION SECTION ---------------------
    # here we reduce the bpm frames which for now is only a cp operation
    mar.mefpp_reduce_bpm(mefpp_dir, static_cal_dir, instrument)

    # we start reducing the biases
    mar.mefpp_reduce_bias(mefpp_dir, set(bias_part_list), instrument)

    # reduction of dome frames
    mar.mefpp_reduce_dome(mefpp_dir, set(dome_part_list), instrument)

    # reduction of sky frames
    mar.mefpp_reduce_sky(mefpp_dir, set(sky_part_list), instrument)
  
    # we reduce the science frames here
    mar.mefpp_reduce_science(mefpp_dir, set(sci_part_list), instrument)
    
    # finally we normalize the image to set the units in e-/s.
    # light correction is applied if the filter are u,g,r o i_sdss
    mar.mefpp_normalize_science(set(sci_part_list), instrument)

print("\n\n\n ------------------- DATA ASSOCIATION AND REDUCTION DONE ----------------------\n\n\n")



#---------------------- DATA ASSOCIATION TREE -------------------------
-------------------------------------------------------------------------
scipart: /Users/fselman/Pictures/OCAM/MultiObject/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
        bpm part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
        bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2017-03-13T09:44:19
        MFF  part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253584/r_SDSS/2017-03-13T23:12:13
         mffpart: /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253584/r_SDSS/2017-03-13T23:12:13
                  bpm  part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
                  bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2017-03-14T09:41:34
                  dome part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/dome/574184/r_SDSS/2017-03-14T09:41:34
                   sky part:  /Users/fselman/Pictures/OCA

 [astropy.io.fits.verify]





#---------------------- DATA ASSOCIATION TREE -------------------------
-------------------------------------------------------------------------
scipart: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/NB_659/2014-12-02T07:04:51
        bpm part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
        bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-02T08:35:57
        MFF  part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253598/NB_659/2014-12-02T23:28:39
         mffpart: /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253598/NB_659/2014-12-02T23:28:39
                  bpm  part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
                  bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-03T08:25:12
                  dome part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/dome/574184/NB_659/2014-12-02T08:35:57
                   sky part:  /Users/fselman/Pictures/OCAM/M



#---------------------- DATA ASSOCIATION TREE -------------------------
-------------------------------------------------------------------------
scipart: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/r_SDSS/2014-12-02T06:51:24
        bpm part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
        bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-02T08:35:57
        MFF  part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253584/r_SDSS/2014-12-02T23:40:05
         mffpart: /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200253584/r_SDSS/2014-12-02T23:40:05
                  bpm  part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
                  bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-03T08:25:12
                  dome part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/dome/574184/r_SDSS/2014-12-02T08:35:57
                   sky part:  /Users/fselman/Pictures/OCAM/M



#---------------------- DATA ASSOCIATION TREE -------------------------
-------------------------------------------------------------------------
scipart: /Users/fselman/Pictures/OCAM/MultiObject/science/M67/g_SDSS/2014-12-02T06:32:53
        bpm part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
        bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-02T08:35:57
        MFF  part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200254042/g_SDSS/2014-11-30T23:28:02
         mffpart: /Users/fselman/Pictures/OCAM/MultiObject/calib/sky/200254042/g_SDSS/2014-11-30T23:28:02
                  bpm  part:  /Users/fselman/Pictures/OCAM/MultiObject/static_calib/bpm
                  bias part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/bias/574184/2014-12-01T09:02:04
                  dome part:  /Users/fselman/Pictures/OCAM/MultiObject/calib/dome/574184/g_SDSS/2014-12-02T08:35:57
                   sky part:  /Users/fselman/Pictures/OCAM/M